In [ ]:
from pathlib import Path

chunk_sizes = [400, 600, 800]
chunk_overlaps = [100, 200, 300]

DATA_DIR = Path("../data")
QDRANT_URL = "http://localhost:6333"

In [ ]:
# === 1. Install dependencies (if not already installed) ===
# !pip install langchain langchain-llms langchain-community qdrant-client openai bs4 PyPDF2

import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# === 2. Configure API Key & Qdrant ===
import sys
sys.path.append(str(Path().resolve().parent))
from backend.app.core.config import OPENAI_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
# === 3. Initialize Qdrant ===
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_size = len(embeddings.embed_query("sample text"))
client = QdrantClient(url=QDRANT_URL)

In [ ]:
# === 4. Read PDF and Split ===
def load_pdfs(data_dir):
    pdf_files = list(data_dir.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDFs.")
    docs = []
    for pdf_path in pdf_files:
        loader = PyPDFLoader(str(pdf_path))
        pages = loader.load()
        for i, page in enumerate(pages):
            meta = {
                "title": pdf_path.stem,
                "pdf_id": pdf_path.stem,
                "page": i + 1
            }
            doc = Document(page_content=page.page_content, metadata=meta)
            docs.append(doc)
    return docs

def chunk_documents(docs, chunk_size=600, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_documents(docs)

raw_docs = load_pdfs(DATA_DIR)

Found 50 PDFs.


In [ ]:
# === 5. Write to Qdrant ===
def safe_add_docs(vec_store, docs):
    safe_docs = []
    for doc in docs:
        safe_meta = {k: str(v) if v is not None else "" for k, v in doc.metadata.items()}
        safe_docs.append(Document(page_content=str(doc.page_content), metadata=safe_meta))
    vec_store.add_documents(safe_docs)

In [ ]:
# === Combine loop parameters to generate multiple collections, named using chunk_size / chunk_overlap ===
for chunk_size in chunk_sizes: 
    for chunk_overlap in chunk_overlaps: 
        collection_name = f"experiment_chunk{chunk_size}_overlap{chunk_overlap}"

        # Delete existing collections to avoid duplicates
        if client.collection_exists(collection_name):
            client.delete_collection(collection_name)
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE)
        )

        chunks = chunk_documents(raw_docs, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        print(f"[{collection_name}] Total chunks: {len(chunks)}")

        vector_store = QdrantVectorStore(
            client=client,
            collection_name=collection_name,
            embedding=embeddings
        )

        safe_add_docs(vector_store, chunks)
        print(f"[{collection_name}] All chunks added ✅")

[experiment_chunk400_overlap100] Total chunks: 6158
[experiment_chunk400_overlap100] All chunks added ✅
[experiment_chunk400_overlap200] Total chunks: 8664
[experiment_chunk400_overlap200] All chunks added ✅
[experiment_chunk400_overlap300] Total chunks: 15715
[experiment_chunk400_overlap300] All chunks added ✅
[experiment_chunk600_overlap100] Total chunks: 3813
[experiment_chunk600_overlap100] All chunks added ✅
[experiment_chunk600_overlap200] Total chunks: 4462
[experiment_chunk600_overlap200] All chunks added ✅
[experiment_chunk600_overlap300] Total chunks: 5553
[experiment_chunk600_overlap300] All chunks added ✅
[experiment_chunk800_overlap100] Total chunks: 2823
[experiment_chunk800_overlap100] All chunks added ✅
[experiment_chunk800_overlap200] Total chunks: 3092
[experiment_chunk800_overlap200] All chunks added ✅
[experiment_chunk800_overlap300] Total chunks: 3468
[experiment_chunk800_overlap300] All chunks added ✅


In [ ]:
# View all collections in Qdrant
collections = client.get_collections().collections
print("All collections in Qdrant:")
for c in collections:
    print("-", c.name)

All collections in Qdrant:
- experiment_chunk400_overlap100
- test
- mini_test
- experiment_chunk600_overlap200
- experiment_chunk600_overlap300
- experiment_chunk800_overlap200
- experiment_chunk800_overlap100
- experiment_chunk800_overlap300
- experiment_chunk400_overlap200
- rag_experiment
- experiment_chunk400_overlap300
- experiment_chunk600_overlap100


In [ ]:
for c in collections:
    collection_name = c.name
    results = client.scroll(
    collection_name=collection_name,
    limit=5
    )
    print(results[:5])

([Record(id='0005ffb7-da81-477f-818c-065710496c95', payload={'page_content': 'REMS. Add (and Authorized Generic) after the corresponding NDA number. \nApplication Holder: [applicant name] Use this only for single-applicant REMS. \nInitial [Shared System] REMS Approval: [MM/YYYY] \nMost Recent REMS Update: [MM/YYYY] Enter the date of the most recent REMS Revision or \napproved Modification. If there are no updates since the initial approval, delete the text. \nII. REMS Goal(s)', 'metadata': {'title': '164344', 'pdf_id': '164344', 'page': '9'}}, vector=None, shard_key=None, order_value=None), Record(id='0011e1a9-6c33-40ec-bb14-11c6e1198603', payload={'page_content': 'where possible, and confirmation of the margin of safety for the labeled dosage \nregimen (dose, route, frequency, duration) in the intended species/class(es). \nThe SR should evaluate a specific formulation of drug produced in accordance \nwith appropriate manufacturing practices.  Formulation should be taken into \nconside